# 🪐 Astro Product Scraper (versi API)

Mengambil semua produk astronauts.id langsung dari API resminya (`requests`, tanpa browser), jadi **cepat** (ratusan produk per detik)
dan **ringan** untuk dijalankan tiap hari.

### Yang sudah dibuktikan dari hasil tes kamu
| Temuan | Bukti | Dipakai di scraper |
|---|---|---|
| Endpoint | `GET api.astronauts.id/api/product/location/591?name=...` | `fetch_page()` |
| Tanpa login | Tes B (tanpa cookie/authorization) → HTTP 200 | tidak perlu akun |
| Header wajib | Hanya `x-device` & `x-device-id` yang bikin gagal bila dibuang | `HEADERS` |
| Paginasi | Respons bergaya Spring: `content`, `totalElements`, `last`; halaman via `pageIndex` mulai 0 | `crawl_keyword()` |
| Ukuran halaman | `pageSize` 100 OK, 200 → HTTP 400 | `PAGE_SIZE = 100` |

### ▶️ Urutan
| Cell | Fungsi |
|---|---|
| 1 | Install |
| 2 | Config |
| 3 | Fungsi API & parser |
| 4 | **Tes cepat** (2 halaman) → cek hasil dulu |
| 5 | Crawl penuh |
| 6 | Export CSV |

### Penjelasan `SWEEP`
Pencarian dengan `name=""` kemungkinan mengembalikan seluruh katalog, tapi itu belum terbukti (tes F hanya menampilkan 20 produk pertama).
Scraper membandingkan jumlah yang terkumpul dengan `totalElements` dari API:
- `auto` (default): kalau kurang dari total, otomatis menyapu kata kunci `a-z` dan `0-9` lalu menggabungkan hasil (dedup lewat `product_id`)
- `always`: selalu menyapu · `never`: tidak pernah

### Perubahan v2: sweep `always` + sapuan lanjutan
API membatasi hasil tiap pencarian ke 10.000 produk teratas. Kata kunci a-z/0-9 bisa mengembalikan 10.000 yang sama, jadi total tidak naik. Mode `always` sekarang menambah **sapuan lanjutan**: kata dari nama produk yang sudah ditemukan (merek, jenis, ukuran) dipakai sebagai kata kunci baru sampai tidak ada produk baru. Perhatikan log: baris bertanda `[TERPOTONG batas API]` menunjukkan pencarian yang kena batas 10.000.


## Cell 1: Install

In [ ]:
!pip install -q requests pandas

## Cell 2: Config

Di Colab, isi `x-device-id` lewat environment (jangan ditulis permanen di notebook yang dibagikan):
```python
os.environ["ASTRO_DEVICE_ID"] = "isi_nilai_x-device-id_dari_DevTools"
```
Jalankan baris itu **sebelum** cell di bawah. Tanpa itu, scraper memakai ID acak (mungkin tetap berhasil, mungkin ditolak server).

In [ ]:
import os
# os.environ["ASTRO_DEVICE_ID"] = "..."   # ← isi lalu hapus tanda #

import json, os, random, re, secrets, sys, time
from collections import Counter
from datetime import datetime, timezone, timedelta

import pandas as pd
import requests

# ── Config ─────────────────────────────────────────────────────────────────
API_BASE     = os.getenv("ASTRO_API_BASE", "https://api.astronauts.id")
SITE_BASE    = "https://www.astronauts.id"
LOCATION_IDS = [int(x) for x in os.getenv("ASTRO_LOCATION_IDS", "591").split(",")]   # 591 = lokasi dari URL kamu
PAGE_SIZE    = int(os.getenv("ASTRO_PAGE_SIZE", "100"))   # hasil tes: 100 OK, 200 ditolak (HTTP 400)
MAX_PAGES    = int(os.getenv("ASTRO_MAX_PAGES", "0"))     # 0 = semua. Isi 2 untuk tes cepat
HARD_CAP     = 1000                                       # pengaman: maks halaman per kata kunci
SWEEP        = os.getenv("ASTRO_SWEEP", "always")         # auto | always | never
DELAY        = (float(os.getenv("ASTRO_DELAY_MIN", "0.4")), float(os.getenv("ASTRO_DELAY_MAX", "1.0")))   # jeda acak antar request (detik)
API_CAP      = int(os.getenv("ASTRO_API_CAP", "10000"))   # batas hasil per pencarian di sisi API (100 halaman x 100)
DEEP         = os.getenv("ASTRO_DEEP", "1") == "1"        # sapuan lanjutan: kata dari nama produk yang sudah ditemukan
MAX_ROUNDS   = int(os.getenv("ASTRO_MAX_ROUNDS", "6"))    # maks ronde sapuan lanjutan
MIN_TOKEN_LEN = int(os.getenv("ASTRO_MIN_TOKEN_LEN", "3"))
TIME_BUDGET_MIN = float(os.getenv("ASTRO_TIME_BUDGET_MIN", "0"))   # 0 = tanpa batas waktu (Colab). Isi mis. 150 di server
BUDGET_HIT   = False
KEYWORDS     = list("abcdefghijklmnopqrstuvwxyz0123456789")

# Dua header ini WAJIB (hasil tes): x-device dan x-device-id. Tanpanya API menjawab HTTP 400 / EC1001.
# x-device-id = ID anonim browser. Simpan di environment variable / GitHub Secret, jangan ditulis di kode.
DEVICE_ID = os.getenv("ASTRO_DEVICE_ID") or secrets.token_hex(32)
HEADERS = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "id-ID,id;q=0.9,en;q=0.8",
    "origin": SITE_BASE,
    "referer": SITE_BASE + "/",
    "user-agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/155.0.0.0 Safari/537.36"),
    "x-api-version": os.getenv("ASTRO_API_VERSION", "1.9.11"),
    "x-device": "web-customer",
    "x-device-id": DEVICE_ID,
    "x-device-version": os.getenv("ASTRO_DEVICE_VERSION", "3.4.9"),
}

OUT_DIR = "data"
WIB     = timezone(timedelta(hours=7))
TODAY   = datetime.now(WIB).strftime("%Y-%m-%d")
os.makedirs(OUT_DIR, exist_ok=True)

HISTORY_FULL = os.getenv("ASTRO_HISTORY_FULL", "0") == "1"   # 1 = history juga menyimpan SEMUA kolom (file cepat membesar)
SAVE_RAW     = os.getenv("ASTRO_SAVE_RAW", "0") == "1"       # 1 = simpan juga JSON mentah per hari (.jsonl.gz), tanpa diubah sama sekali

print(f"Config OK | {TODAY} | lokasi {LOCATION_IDS} | pageSize {PAGE_SIZE}")
if not os.getenv("ASTRO_DEVICE_ID"):
    print("ℹ ASTRO_DEVICE_ID belum di-set → memakai ID acak. Kalau muncul error EC1001, isi dengan x-device-id dari browser.")

## Cell 3: Fungsi API & parser: **semua field API diambil**

Setiap produk diratakan (`flatten`) tanpa membuang field apa pun:

| Bentuk di API | Jadi di CSV |
|---|---|
| nilai biasa (`brandId`, `remaining_qty`, ...) | kolom dengan nama asli API |
| objek (`product_rating`) | kolom berawalan: `product_rating_rating_score`, `product_rating_is_show_rating` |
| list angka/teks (`categoryIds`) | digabung dengan `\|`, mis. `286\|293\|1468` (bisa di-Split di Power Query) |
| list objek (`product_images`, `masterVariantVariants`) | teks JSON dalam satu sel |

Tambahan kolom turunan di depan: `product_url`, `pack_size`, `normal_price`, `discount_price`, `selling_price` (harga yang benar-benar dibayar), `discount_pct_calc`.

In [ ]:
# ── Ambil data dari API ────────────────────────────────────────────────────
class ApiError(Exception):
    pass


RAW = {}   # (location_id, product_id) -> item JSON mentah


def to_int(v):
    s = re.sub(r"[^\d]", "", str(v)) if v not in (None, "") else ""
    return int(s) if s else None


def guess_pack_size(name):
    ms = re.findall(r"(\d+(?:[.,]\d+)?)\s?(ml|liter|lt|kg|gr|g|l|pcs|pc|sachet|lembar|roll)\b", name or "", re.I)
    return f"{ms[-1][0]}{ms[-1][1].lower()}" if ms else None


def flatten(d, prefix=""):
    """Ratakan JSON bersarang jadi satu baris datar. Tidak ada field yang dibuang."""
    out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(flatten(v, key + "_"))
        elif isinstance(v, list):
            if not v:
                out[key] = ""
            elif all(not isinstance(x, (dict, list)) for x in v):
                out[key] = "|".join(str(x) for x in v)
            else:
                out[key] = json.dumps(v, ensure_ascii=False, separators=(",", ":"))
        else:
            out[key] = v
    return out


def fetch_page(session, location_id, name, page_index):
    url = f"{API_BASE}/api/product/location/{location_id}"
    params = {"name": name, "pageIndex": page_index, "pageSize": PAGE_SIZE, "searchmode": 1,
              "locationIds": location_id, "source": "srp", "layout_type": "grid", "items_per_row": 3}
    for attempt in range(4):
        try:
            r = session.get(url, params=params, timeout=30)
        except requests.RequestException:
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(5 * (attempt + 1))
            continue
        raise ApiError(f"HTTP {r.status_code}: {r.text[:200]}")
    raise ApiError("gagal setelah 4 percobaan (jaringan / server sibuk)")


def parse_item(it, location_id):
    """Kolom turunan di depan, lalu SEMUA field asli API (urut abjad)."""
    price = to_int(it.get("product_price"))
    disc  = to_int(it.get("product_discount_price"))
    name  = it.get("product_name")
    slug  = it.get("product_slug")
    derived = {
        "scrape_date": TODAY,
        "location_id": location_id,
        "product_id": it.get("product_id"),
        "product_name": name,
        "product_url": f"{SITE_BASE}/p/{slug}" if slug else None,
        "pack_size": (it.get("product_grammation_label") or "").strip() or guess_pack_size(name),
        "normal_price": price,
        "discount_price": disc,
        "selling_price": disc if disc else price,
        "discount_pct_calc": round((1 - disc / price) * 100, 1) if (price and disc) else 0,
    }
    flat = flatten(it)
    rest = {k: flat[k] for k in sorted(flat) if k not in derived}
    return {**derived, **rest}


def crawl_keyword(session, location_id, name, found, expect=None):
    """Telusuri halaman untuk satu kata kunci. Return (totalElements, produk_baru, selesai_normal).

    expect = jumlah produk yang SUDAH kita punya untuk kata ini. Kalau totalElements <= expect, tidak mungkin
    ada produk baru, jadi halaman berikutnya dilewati (hemat request).
    Halaman > 0 yang ditolak API (HTTP 400, di luar jendela hasil) dianggap batas API, bukan error fatal."""
    page, new, total = 0, 0, None
    while True:
        try:
            data = fetch_page(session, location_id, name, page)
        except ApiError as e:
            if page == 0:
                raise
            print(f"     ! '{name}' halaman {page} ditolak ({str(e)[:60]}) → dianggap batas jendela API", flush=True)
            return total, new, False
        content = data.get("content") or []
        total   = data.get("totalElements", total)
        for it in content:
            pid = it.get("product_id")
            if pid is not None and (location_id, pid) not in found:
                found[(location_id, pid)] = parse_item(it, location_id)
                RAW[(location_id, pid)] = it
                new += 1
        page += 1
        if page == 1 and expect is not None and total is not None and total <= expect:
            return total, new, True
        if data.get("last", True) or not content:
            return total, new, True
        if (MAX_PAGES and page >= MAX_PAGES) or page >= HARD_CAP:
            return total, new, False
        time.sleep(random.uniform(*DELAY))


# ── Sapuan ─────────────────────────────────────────────────────────────────
def _capped(total, complete):
    """True kalau hasil pencarian terpotong batas API (produk di luar jendela tidak terlihat)."""
    return (not complete) or (total is not None and total >= API_CAP)


def _over_budget(t0):
    global BUDGET_HIT
    if TIME_BUDGET_MIN and (time.time() - t0) / 60 > TIME_BUDGET_MIN:
        BUDGET_HIT = True
        return True
    return False


def _loc_count(found, loc):
    return sum(1 for k in found if k[0] == loc)


def _words(name):
    return set(re.findall(r"[a-z0-9]+", (name or "").lower()))


def _keyword_sweep(session, loc, found, queried, t0):
    """Tahap 1: kata kunci a-z dan 0-9."""
    for kw in KEYWORDS:
        if _over_budget(t0):
            print("  ! batas waktu tercapai, sapuan dihentikan", flush=True)
            return False
        names = [(v.get("product_name") or "").lower() for k, v in found.items() if k[0] == loc]
        expect = sum(1 for n in names if kw in n)
        total, new, complete = crawl_keyword(session, loc, kw, found, expect=expect)
        queried.add(kw)
        flag = "  [terpotong batas API]" if _capped(total, complete) else ""
        print(f"     '{kw}': total API {total} | +{new:<4} → {_loc_count(found, loc)}{flag}", flush=True)
        time.sleep(random.uniform(*DELAY))
    return True


def _deep_sweep(session, loc, found, queried, t0):
    """Tahap 2: kata dari nama produk yang sudah ditemukan, diulang sampai tidak ada produk baru."""
    for rnd in range(1, MAX_ROUNDS + 1):
        counter = Counter(w for k, v in found.items() if k[0] == loc for w in _words(v.get("product_name")))
        todo = [w for w, c in counter.most_common()
                if len(w) >= MIN_TOKEN_LEN and w not in queried and c < API_CAP]
        if not todo:
            print(f"  ronde {rnd}: tidak ada kata baru", flush=True)
            return True
        before = _loc_count(found, loc)
        print(f"  ronde {rnd}: {len(todo)} kata kunci dari nama produk", flush=True)
        for i, w in enumerate(todo, 1):
            if _over_budget(t0):
                print(f"  ! batas waktu tercapai di ronde {rnd} ({i}/{len(todo)}), sapuan dihentikan", flush=True)
                return False
            crawl_keyword(session, loc, w, found, expect=counter[w])
            queried.add(w)
            if i % 200 == 0:
                print(f"     {i}/{len(todo)} kata | total {_loc_count(found, loc)}", flush=True)
            time.sleep(random.uniform(*DELAY))
        gained = _loc_count(found, loc) - before
        print(f"  ronde {rnd} selesai: +{gained} produk → {_loc_count(found, loc)}", flush=True)
        if gained == 0:
            return True
    print(f"  ! berhenti di batas {MAX_ROUNDS} ronde (naikkan ASTRO_MAX_ROUNDS jika masih bertambah)", flush=True)
    return True


def run_crawl():
    session = requests.Session()
    session.headers.update(HEADERS)
    found = {}
    t0 = time.time()
    for loc in LOCATION_IDS:
        print(f"\n── Lokasi {loc}", flush=True)
        total, new, complete = crawl_keyword(session, loc, "", found)
        got = _loc_count(found, loc)
        capped = _capped(total, complete)
        print(f"  name=''  → totalElements={total} | terkumpul {got} | selesai normal={complete}"
              + ("  [TERPOTONG batas API]" if capped else ""), flush=True)

        incomplete = capped or total is None or got < total
        if SWEEP == "always" or (SWEEP == "auto" and incomplete):
            queried = set()
            print(f"  → sapuan kata kunci ({len(KEYWORDS)} kata)", flush=True)
            ok = _keyword_sweep(session, loc, found, queried, t0)
            if ok and DEEP:
                print("  → sapuan lanjutan (kata dari nama produk)", flush=True)
                _deep_sweep(session, loc, found, queried, t0)
        print(f"  ✓ lokasi {loc}: {_loc_count(found, loc)} produk unik "
              f"({(time.time() - t0) / 60:.1f} menit)", flush=True)
    return found


## Cell 4: Tes cepat (2 halaman)

Pastikan kolom terisi benar sebelum crawl penuh. Perhatikan angka `totalElements` yang dicetak: itu ukuran katalog menurut API.

In [ ]:
_MP, _SW = MAX_PAGES, SWEEP
MAX_PAGES, SWEEP = 2, "never"
test = run_crawl()
MAX_PAGES, SWEEP = _MP, _SW
pd.DataFrame(list(test.values())).drop(columns=["image_url"]).head(10)

## Cell 5: Crawl penuh

Kalau `ApiError` / HTTP 400 muncul, isi `ASTRO_DEVICE_ID` (Cell 2) dengan nilai dari browser.

In [ ]:
found = run_crawl()

## Cell 6: Export

`astro_products_latest.csv` = **semua kolom**, ditimpa tiap hari. `astro_products_history.csv` = ditumpuk per `scrape_date` untuk tren; kolom berat (gambar, ikon, `categoryIds`) dilewati agar file tidak membengkak. Set `ASTRO_HISTORY_FULL=1` untuk menyimpan semuanya, atau `ASTRO_SAVE_RAW=1` untuk menyimpan JSON mentah per hari.

In [ ]:
# ── Export CSV ─────────────────────────────────────────────────────────────
FIRST = ["scrape_date", "location_id", "product_id", "product_name", "product_url", "pack_size",
         "normal_price", "discount_price", "selling_price", "discount_pct_calc"]
NUMERIC = ["product_id", "normal_price", "discount_price", "selling_price", "product_price", "product_discount_price"]
HEAVY_NAMES = {"categoryIds", "bottomsheets", "masterVariantVariants", "variant_relatives_details", "product_images"}


def is_heavy(col):
    c = col.lower()
    return col in HEAVY_NAMES or "image" in c or "icon" in c or "blur_hash" in c or c.startswith("product_reminder")


def export(found):
    df = pd.DataFrame(list(found.values()))
    df = df[FIRST + sorted(c for c in df.columns if c not in FIRST)]          # urutan kolom stabil antar hari
    for c in NUMERIC:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").round().astype("Int64")
    df = df.sort_values(["location_id", "product_id"]).reset_index(drop=True)

    # 1) latest: SEMUA kolom, ditimpa tiap hari
    df.to_csv(f"{OUT_DIR}/astro_products_latest.csv", index=False, encoding="utf-8-sig")

    # 2) history: ditumpuk tiap hari. Kolom berat (gambar, ikon, categoryIds, dst.) dilewati kecuali ASTRO_HISTORY_FULL=1
    heavy = [] if HISTORY_FULL else [c for c in df.columns if is_heavy(c) and c not in FIRST]
    lean = df.drop(columns=heavy).astype("string")
    hist = f"{OUT_DIR}/astro_products_history.csv"
    if os.path.exists(hist):
        old = pd.read_csv(hist, dtype="string")
        old = old[old["scrape_date"] != TODAY]
        lean = pd.concat([old, lean], ignore_index=True)
    lean.to_csv(hist, index=False, encoding="utf-8-sig")

    # 3) opsional: JSON mentah persis dari API (tanpa perubahan apa pun)
    if SAVE_RAW:
        import gzip
        os.makedirs(f"{OUT_DIR}/raw", exist_ok=True)
        with gzip.open(f"{OUT_DIR}/raw/astro_raw_{TODAY}.jsonl.gz", "wt", encoding="utf-8") as f:
            for (loc, pid), it in RAW.items():
                f.write(json.dumps({"scrape_date": TODAY, "location_id": loc, "item": it}, ensure_ascii=False) + "\n")

    n_disc = int(df["discount_price"].notna().sum())
    print(f"✓ {len(df)} produk × {df.shape[1]} kolom | {n_disc} sedang diskon")
    print(f"  latest: semua {df.shape[1]} kolom | history: {lean.shape[1]} kolom"
          + (f" (dilewati {len(heavy)} kolom berat)" if heavy else " (semua kolom)"))
    return df

df = export(found)
display(df.head())

try:
    from google.colab import files as colab_files
    colab_files.download(f"{OUT_DIR}/astro_products_latest.csv")
except ImportError:
    pass

## 🔁 Otomatis tiap hari

Gunakan `astro_api_scraper.py` + `.github/workflows/daily_scrape.yml` (GitHub Actions, 06:00 WIB), atau Windows Task Scheduler:
```
schtasks /create /sc daily /st 06:00 /tn "AstroScrape" /tr "python C:\\path\\astro_api_scraper.py"
```
Simpan `x-device-id` sebagai **GitHub Secret** bernama `ASTRO_DEVICE_ID` (Settings → Secrets and variables → Actions).